# Powder Bed Fusion — Individual Build Visualizations
**Gas Sensor RAW Values & Power_W · Interactive Plotly Figures**

| | Run 1 | Run 2 |
|---|---|---|
| **Start** | 2025-12-15 09:38:22 | 2025-12-16 11:50:09 |
| **End** | 2025-12-16 07:40:16 | 2025-12-16 22:44:29 |
| **Duration** | ~22 hrs | ~11 hrs |

Each build is shown as its own standalone interactive figure.  
X-axis ticks are placed at every whole build hour for easy reading.

---
## 1 · Imports & Configuration

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os

# Render figures inline inside the notebook
pio.renderers.default = "notebook"

# ── Check kaleido is available for static export ──────────────────────────────
try:
    import kaleido
    KALEIDO_OK = True
    print("✓ kaleido available — PDF/PNG export enabled.")
except ImportError:
    KALEIDO_OK = False
    print("⚠ kaleido not found. Install with: pip install kaleido")
    print("  HTML export will still work; PDF/PNG export will be skipped.")

# ── Export settings (edit here) ───────────────────────────────────────────────
OUTPUT_DIR    = "exports"          # folder where all exports are saved
EXPORT_WIDTH  = 1600               # pixels wide  (increase for wider figures)
EXPORT_HEIGHT = 550                # pixels tall
EXPORT_SCALE  = 2                  # 2 = 2x pixel density → 3200x1100 for PNG
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── User-adjustable parameters ────────────────────────────────────────────────
DB_PATH      = "DoE_data.db"   # Path to SQLite database
TABLE_NAME   = "DoERun"       # Table name
DOWNSAMPLE_S = 120             # Resample interval in seconds (2 min → clean plots)

BUILD1_START = "2025-12-15 09:53:09"
BUILD1_END   = "2025-12-16 07:40:16"
BUILD2_START = "2025-12-16 11:50:09"
BUILD2_END   = "2025-12-16 22:44:29"

GAS_COLS  = ["ArgonFlowRAW", "LowFlowRAW", "HighFlowRAW"]
POWER_COL = "Power_W"
ALL_COLS  = GAS_COLS + [POWER_COL]

# ── Visual theme ──────────────────────────────────────────────────────────────
BG       = "#ffffff"
PANEL_BG = "#f5f5f5"
GRID     = "#e0e0e0"
TEXT     = "#1a1a2e"
AXIS_LBL = "#4a5568"

PALETTE = {
    "ArgonFlowRAW" : "#9b5de5",
    "LowFlowRAW"   : "#00b4d8",
    "HighFlowRAW"  : "#06d6a0",
    "Power_W"      : "#ef476f",
}

AXIS_COMMON = dict(
    showgrid    = True,
    gridcolor   = GRID,
    gridwidth   = 1,
    zerolinecolor = "#2a3040",
    tickcolor   = "#4a5568",
    linecolor   = "#cccccc",
    tickfont    = dict(color=TEXT, size=11),
    title_font  = dict(color=AXIS_LBL, size=12),
)

print("✓ Configuration loaded.")
print(f"✓ Exports will be saved to: {os.path.abspath(OUTPUT_DIR)}/")

---
## 2 · Load & Prepare Data

In [ ]:
def load_build(db_path, table, start, end):
    """Query one build window from the SQLite database."""
    sql = f"""
        SELECT Timestamp, ArgonFlowRAW, LowFlowRAW, HighFlowRAW, Power_W
        FROM   {table}
        WHERE  Timestamp BETWEEN '{start}' AND '{end}'
        ORDER  BY Timestamp
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(sql, conn, parse_dates=["Timestamp"])
    return df


def prepare_build(df, resample_s):
    """
    Resample to reduce visual clutter and add a
    'build_hours' column anchored to the first timestamp.
    """
    t0 = df["Timestamp"].iloc[0]
    ds = (
        df.set_index("Timestamp")
          .resample(f"{resample_s}s")
          .mean()
          .reset_index()
    )
    ds["build_hours"] = (ds["Timestamp"] - t0).dt.total_seconds() / 3600
    return ds, t0


# Load raw data
b1_raw = load_build(DB_PATH, TABLE_NAME, BUILD1_START, BUILD1_END)
b2_raw = load_build(DB_PATH, TABLE_NAME, BUILD2_START, BUILD2_END)

# Downsample
b1, b1_t0 = prepare_build(b1_raw, DOWNSAMPLE_S)
b2, b2_t0 = prepare_build(b2_raw, DOWNSAMPLE_S)

b1_dur = b1["build_hours"].max()
b2_dur = b2["build_hours"].max()

print(f"Run 1 — raw rows: {len(b1_raw):>7,}  →  downsampled: {len(b1):>4,}  |  duration: {b1_dur:.2f} hrs")
print(f"Run 2 — raw rows: {len(b2_raw):>7,}  →  downsampled: {len(b2):>4,}  |  duration: {b2_dur:.2f} hrs")
print(f"Resample interval : {DOWNSAMPLE_S} s = every {DOWNSAMPLE_S//60} min")

---
## 3 · Shared Figure Builder
A single reusable function that draws the dual-axis interactive figure for any build.

In [ ]:
def build_figure(df, build_label, start_str, end_str, duration_hrs):
    """
    Create an interactive Plotly figure for one build.

    Layout
    ------
    - Left  Y-axis : Gas sensor RAW values (ArgonFlowRAW, LowFlowRAW, HighFlowRAW)
    - Right Y-axis : Power_W
    - X-axis       : Build hours, ticked at every whole hour
    """

    # Whole-hour tick values (0, 1, 2 … ceil(duration))
    max_hour    = int(np.ceil(duration_hrs))
    hour_ticks  = list(range(0, max_hour + 1))
    hour_labels = [str(h) for h in hour_ticks]

    fig = make_subplots(
        specs     = [[{"secondary_y": True}]],
        subplot_titles = [" "],          # placeholder — real title via layout
    )

    # ── Gas RAW traces (left axis) ────────────────────────────────────────────
    for col in GAS_COLS:
        fig.add_trace(
            go.Scattergl(
                x    = df["build_hours"],
                y    = df[col],
                name = col,
                mode = "lines",
                line = dict(color=PALETTE[col], width=1.8),
                hovertemplate = (
                    f"<b>{col}</b><br>"
                    "Hour: %{x:.2f}<br>"
                    "RAW: %{y:.1f}<extra></extra>"
                ),
            ),
            secondary_y = False,
        )

    # ── Power trace (right axis) ──────────────────────────────────────────────
    fig.add_trace(
        go.Scattergl(
            x    = df["build_hours"],
            y    = df[POWER_COL],
            name = "Power_W",
            mode = "lines",
            line = dict(color=PALETTE[POWER_COL], width=1.6),
            hovertemplate = (
                "<b>Power_W</b><br>"
                "Hour: %{x:.2f}<br>"
                "Power: %{y:.0f} W<extra></extra>"
            ),
        ),
        secondary_y = True,
    )

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title = dict(
            text = (
                f"<b>{build_label}</b>  ·  "
                f"Gas Sensor RAW & Power_W<br>"
                f"<sup>{start_str}  →  {end_str}  "
                f"({duration_hrs:.2f} hrs)  ·  "
                f"{DOWNSAMPLE_S//60}-min avg downsampled</sup>"
            ),
            font      = dict(color=TEXT, size=14),
            x         = 0.01,
            xanchor   = "left",
            pad       = dict(t=4),
        ),
        plot_bgcolor  = BG,
        paper_bgcolor = BG,
        font          = dict(color=TEXT, family="Segoe UI, Arial, sans-serif", size=12),
        hovermode     = "x unified",
        hoverlabel    = dict(
            bgcolor    = PANEL_BG,
            bordercolor= "#3a4560",
            font       = dict(size=11, color=TEXT),
        ),
        legend = dict(
            bgcolor      = PANEL_BG,
            bordercolor  = "#2a3040",
            borderwidth  = 1,
            font         = dict(size=11),
            orientation  = "h",
            x            = 0,
            y            = -0.20,
            itemclick    = "toggle",
            itemdoubleclick = "toggleothers",
        ),
        margin = dict(l=72, r=80, t=90, b=110),
        height = 540,
    )

    # ── X-axis: every whole hour ──────────────────────────────────────────────
    fig.update_xaxes(
        title_text  = "Build Hours",
        range       = [-0.1, max_hour + 0.2],
        tickmode    = "array",
        tickvals    = hour_ticks,
        ticktext    = hour_labels,
        tickangle   = 0,
        showticklabels = True,
        minor       = dict(
            tickvals  = [h + 0.5 for h in hour_ticks[:-1]],
            gridcolor = "#161c28",
            gridwidth = 1,
            showgrid  = True,
        ),
        **AXIS_COMMON,
    )

    # ── Left Y-axis: Gas RAW ──────────────────────────────────────────────────
    # Compute a tight but padded range from all gas columns
    gas_min = df[GAS_COLS].min().min()
    gas_max = df[GAS_COLS].max().max()
    gas_pad = (gas_max - gas_min) * 0.08
    fig.update_yaxes(
        title_text  = "Gas Sensor RAW (ADC counts)",
        range       = [gas_min - gas_pad, gas_max + gas_pad],
        secondary_y = False,
        **AXIS_COMMON,
    )

    # ── Right Y-axis: Power ───────────────────────────────────────────────────
    pwr_max = df[POWER_COL].max()
    fig.update_yaxes(
        title_text  = "Power_W  (W)",
        range       = [0, pwr_max * 1.12],
        showgrid    = False,
        secondary_y = True,
        tickcolor   = "#4a5568",
        linecolor   = "#2a3040",
        tickfont    = dict(color=TEXT, size=11),
        title_font  = dict(color=AXIS_LBL, size=12),
    )

    # ── Whole-hour vertical reference lines ───────────────────────────────────
    for h in hour_ticks[1:]:
        fig.add_vline(
            x          = h,
            line_dash  = "dot",
            line_color = "#2a3040",
            line_width = 1,
            opacity    = 0.6,
        )

    return fig


print("✓ Figure builder defined.")

In [ ]:
def export_fig(fig, filename_stem, w=None, h=None, scale=None):
    """
    Save a Plotly figure in three formats:
      - PDF  : vector, perfect quality at any zoom in LaTeX/Overleaf
      - PNG  : high-res raster (scale * w x scale * h pixels)
      - HTML : fully interactive standalone file

    Parameters
    ----------
    fig           : plotly Figure object
    filename_stem : base name without extension, e.g. "run1_overview"
    w, h, scale   : override EXPORT_WIDTH / EXPORT_HEIGHT / EXPORT_SCALE
    """
    w     = w     or EXPORT_WIDTH
    h     = h     or EXPORT_HEIGHT
    scale = scale or EXPORT_SCALE

    html_path = os.path.join(OUTPUT_DIR, f"{filename_stem}.html")
    fig.write_html(html_path, include_plotlyjs="cdn")
    print(f"  ✓ HTML  → {html_path}")

    if KALEIDO_OK:
        pdf_path = os.path.join(OUTPUT_DIR, f"{filename_stem}.pdf")
        fig.write_image(pdf_path, width=w, height=h, scale=1)
        print(f"  ✓ PDF   → {pdf_path}  (vector, recommended for LaTeX)")

        png_path = os.path.join(OUTPUT_DIR, f"{filename_stem}.png")
        fig.write_image(png_path, width=w, height=h, scale=scale)
        print(f"  ✓ PNG   → {png_path}  ({w*scale}x{h*scale}px)")
    else:
        print("  ⚠ PDF/PNG skipped — install kaleido: pip install kaleido")

print("✓ export_fig() helper defined.")

---
## 4 · Build 1  ·  2025-12-15 09:38 → 2025-12-16 07:40  (~22 hrs)

In [ ]:
fig_b1 = build_figure(
    df            = b1,
    build_label   = "Run 1",
    start_str     = BUILD1_START,
    end_str       = BUILD1_END,
    duration_hrs  = b1_dur,
)
fig_b1.show()

print("Exporting Run 1 overview...")
export_fig(fig_b1, "run1_overview")

> **Tips:** Click legend items to toggle individual channels on/off.  
> Double-click a legend item to isolate it. Use the toolbar to zoom, pan or export PNG.

---
## 5 · Build 2  ·  2025-12-16 11:50 → 2025-12-16 22:44  (~11 hrs)

In [ ]:
fig_b2 = build_figure(
    df            = b2,
    build_label   = "Run 2",
    start_str     = BUILD2_START,
    end_str       = BUILD2_END,
    duration_hrs  = b2_dur,
)
fig_b2.show()

print("Exporting Run 2 overview...")
export_fig(fig_b2, "run2_overview")

> **Tips:** Click legend items to toggle individual channels on/off.  
> Double-click a legend item to isolate it. Use the toolbar to zoom, pan or export PNG.

---
## 6 · Summary Statistics

In [ ]:
rows = []
for label, df in [("Run 1", b1), ("Run 2", b2)]:
    for col in ALL_COLS:
        rows.append({
            "Run"    : label,
            "Channel"  : col,
            "Min"      : round(df[col].min(),  1),
            "Mean"     : round(df[col].mean(), 1),
            "Max"      : round(df[col].max(),  1),
            "Std Dev"  : round(df[col].std(),  1),
        })

stats = pd.DataFrame(rows).set_index(["Run", "Channel"])
stats.style.background_gradient(subset=["Min","Mean","Max"], cmap="Blues")

---
## 7 · Export Figures to HTML (Optional)

In [ ]:
# ── All exports are already saved inline after each figure cell.
# ── This cell lists everything that was exported. ────────────────────────────
import glob
exported = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*")))
print(f"All files in {OUTPUT_DIR}/:
")
for f in exported:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<45}  {size_kb:>8.1f} KB")
print(f"
Total: {len(exported)} files")
print("
Tip: Use the .pdf files in LaTeX with \includegraphics[width=\linewidth]{filename}")

In [ ]:
# ── Purge & Exposure time windows ─────────────────────────────────────────────
PURGE_R1_START    = "2025-12-15 09:53:09"
PURGE_R1_END      = "2025-12-15 10:21:50"
EXPOSURE_R1_START = "2025-12-15 10:21:50"
EXPOSURE_R1_END   = "2025-12-16 07:54:55"

PURGE_R2_START    = "2025-12-16 12:04:56"
PURGE_R2_END      = "2025-12-16 12:21:51"
EXPOSURE_R2_START = "2025-12-16 12:21:51"
EXPOSURE_R2_END   = "2025-12-16 22:44:21"

print("✓ Purge & Exposure windows defined.")

In [ ]:
def load_segment(db_path, table, start, end, anchor_start):
    """
    Load a time window at full resolution and compute build_hours
    relative to anchor_start (the build start timestamp).
    """
    sql = f"""
        SELECT Timestamp, ArgonFlowRAW, LowFlowRAW, HighFlowRAW, Power_W
        FROM   {table}
        WHERE  Timestamp BETWEEN '{start}' AND '{end}'
        ORDER  BY Timestamp
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(sql, conn, parse_dates=["Timestamp"])

    anchor = pd.Timestamp(anchor_start)
    df["build_hours"] = (df["Timestamp"] - anchor).dt.total_seconds() / 3600
    return df


# Run 1 anchor = build start used in the main notebook
R1_ANCHOR = BUILD1_START
R2_ANCHOR = BUILD2_START

purge_r1    = load_segment(DB_PATH, TABLE_NAME, PURGE_R1_START,    PURGE_R1_END,    R1_ANCHOR)
exposure_r1 = load_segment(DB_PATH, TABLE_NAME, EXPOSURE_R1_START, EXPOSURE_R1_END, R1_ANCHOR)
purge_r2    = load_segment(DB_PATH, TABLE_NAME, PURGE_R2_START,    PURGE_R2_END,    R2_ANCHOR)
exposure_r2 = load_segment(DB_PATH, TABLE_NAME, EXPOSURE_R2_START, EXPOSURE_R2_END, R2_ANCHOR)

for name, seg in [("Purge R1", purge_r1), ("Exposure R1", exposure_r1),
                   ("Purge R2", purge_r2), ("Exposure R2", exposure_r2)]:
    dur = (seg["Timestamp"].max() - seg["Timestamp"].min()).total_seconds() / 60
    print(f"{name:<14} — {len(seg):>6,} rows  |  "
          f"build hours {seg['build_hours'].min():.3f} → {seg['build_hours'].max():.3f}  "
          f"({dur:.1f} min)")

In [ ]:
def build_segment_figure(df, title, subtitle):
    """
    Full-resolution dual-axis interactive figure for a single segment.
    X-axis ticks are placed at every 0.1 build-hour interval (6 min)
    for short windows, or every whole hour for long windows.
    """
    duration = df["build_hours"].max() - df["build_hours"].min()
    x_start  = df["build_hours"].min()
    x_end    = df["build_hours"].max()

    # Adaptive tick spacing: fine for short purge, coarse for long exposure
    if duration <= 1.0:
        tick_step   = 0.1          # every 6 min for short windows
    elif duration <= 4.0:
        tick_step   = 0.25         # every 15 min
    else:
        tick_step   = 1.0          # every hour for long exposures

    tick_vals  = list(np.arange(
        np.floor(x_start / tick_step) * tick_step,
        x_end + tick_step,
        tick_step
    ))
    tick_vals  = [round(t, 4) for t in tick_vals]
    tick_labels = [f"{t:.2f}" if tick_step < 1 else f"{int(round(t))}" for t in tick_vals]

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # ── Gas RAW traces (left axis) ────────────────────────────────────────────
    for col in GAS_COLS:
        fig.add_trace(
            go.Scattergl(
                x    = df["build_hours"],
                y    = df[col],
                name = col,
                mode = "lines",
                line = dict(color=PALETTE[col], width=1.6),
                hovertemplate=(
                    f"<b>{col}</b><br>"
                    "Hour: %{x:.3f}<br>"
                    "RAW: %{y:.1f}<extra></extra>"
                ),
            ),
            secondary_y=False,
        )

    # ── Power trace (right axis) ──────────────────────────────────────────────
    fig.add_trace(
        go.Scattergl(
            x    = df["build_hours"],
            y    = df[POWER_COL],
            name = "Power_W",
            mode = "lines",
            line = dict(color=PALETTE[POWER_COL], width=1.6),
            hovertemplate=(
                "<b>Power_W</b><br>"
                "Hour: %{x:.3f}<br>"
                "Power: %{y:.0f} W<extra></extra>"
            ),
        ),
        secondary_y=True,
    )

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        title=dict(
            text=(
                f"<b>{title}</b><br>"
                f"<sup>{subtitle}</sup>"
            ),
            font=dict(color=TEXT, size=14),
            x=0.01, xanchor="left",
        ),
        plot_bgcolor  = BG,
        paper_bgcolor = BG,
        font          = dict(color=TEXT, family="Segoe UI, Arial, sans-serif", size=12),
        hovermode     = "x unified",
        hoverlabel    = dict(bgcolor=PANEL_BG, bordercolor="#3a4560",
                             font=dict(size=11, color=TEXT)),
        legend=dict(
            bgcolor     = PANEL_BG,
            bordercolor = "#2a3040",
            borderwidth = 1,
            font        = dict(size=11),
            orientation = "h",
            x=0, y=-0.22,
            itemclick         = "toggle",
            itemdoubleclick   = "toggleothers",
        ),
        margin = dict(l=72, r=80, t=90, b=110),
        height = 500,
    )

    # ── X-axis ────────────────────────────────────────────────────────────────
    fig.update_xaxes(
        title_text = "Build Hours",
        range      = [x_start - tick_step * 0.3, x_end + tick_step * 0.3],
        tickmode   = "array",
        tickvals   = tick_vals,
        ticktext   = tick_labels,
        tickangle  = -35,
        **AXIS_COMMON,
    )

    # ── Left Y-axis: Gas RAW ──────────────────────────────────────────────────
    gas_min = df[GAS_COLS].min().min()
    gas_max = df[GAS_COLS].max().max()
    pad     = (gas_max - gas_min) * 0.08
    fig.update_yaxes(
        title_text  = "Gas Sensor RAW (ADC counts)",
        range       = [gas_min - pad, gas_max + pad],
        secondary_y = False,
        **AXIS_COMMON,
    )

    # ── Right Y-axis: Power ───────────────────────────────────────────────────
    pwr_max = df[POWER_COL].max()
    fig.update_yaxes(
        title_text  = "Power_W  (W)",
        range       = [0, pwr_max * 1.12],
        showgrid    = False,
        secondary_y = True,
        tickcolor   = "#4a5568",
        linecolor   = "#2a3040",
        tickfont    = dict(color=TEXT, size=11),
        title_font  = dict(color=AXIS_LBL, size=12),
    )

    # ── Tick reference lines ──────────────────────────────────────────────────
    for t in tick_vals:
        fig.add_vline(
            x=t, line_dash="dot",
            line_color="#2a3040", line_width=1, opacity=0.5,
        )

    return fig


print("✓ Segment figure builder defined.")

In [ ]:
fig_purge_r1 = build_segment_figure(
    df       = purge_r1,
    title    = "Run 1 — Purge Step",
    subtitle = f"{PURGE_R1_START}  →  {PURGE_R1_END}  "
               f"({(purge_r1['build_hours'].max() - purge_r1['build_hours'].min())*60:.1f} min)  ·  Full resolution",
)
fig_purge_r1.show()

print("Exporting run1 purge...")
export_fig(fig_purge_r1, "run1_purge")

In [ ]:
# ── How many minutes from exposure start to show ──────────────────────────────
PREVIEW_MINUTES = 40   # change to 30 if you prefer

exposure_r1_preview = exposure_r1[
    exposure_r1["build_hours"] <= exposure_r1["build_hours"].iloc[0] + PREVIEW_MINUTES / 60
].copy()

fig_exposure_r1 = build_segment_figure(
    df       = exposure_r1_preview,
    title    = "Run 1 — Exposure Step (first few layers)",
    subtitle = #f"{EXPOSURE_R1_START}  →  "
               #f"{exposure_r1_preview['Timestamp'].max()}  ·  "
               f"{EXPOSURE_R1_START} → {EXPOSURE_R1_END} . "
               f"Zoomed Version ·  Full Resolution",
)
fig_exposure_r1.show()

print("Exporting run1 exposure...")
export_fig(fig_exposure_r1, "run1_exposure")

In [ ]:
fig_purge_r2 = build_segment_figure(
    df       = purge_r2,
    title    = "Run 2 — Purge Step",
    subtitle = f"{PURGE_R2_START}  →  {PURGE_R2_END}  "
               f"({(purge_r2['build_hours'].max() - purge_r2['build_hours'].min())*60:.1f} min)  ·  Full resolution",
)
fig_purge_r2.show()

print("Exporting run2 purge...")
export_fig(fig_purge_r2, "run2_purge")

In [ ]:
exposure_r2_preview = exposure_r2[
    exposure_r2["build_hours"] <= exposure_r2["build_hours"].iloc[0] + PREVIEW_MINUTES / 60
].copy()

fig_exposure_r2 = build_segment_figure(
    df       = exposure_r2_preview,
    title    = "Run 2 — Exposure Step (first few layers)",
    subtitle = #f"{EXPOSURE_R2_START}  →  "
               #f"{exposure_r2_preview['Timestamp'].max()}  ·  "
               f"{EXPOSURE_R2_START} → {EXPOSURE_R2_END} . "
               f"Zoomed Version · Full Resolution",
)
fig_exposure_r2.show()

print("Exporting run2 exposure...")
export_fig(fig_exposure_r2, "run2_exposure")

In [ ]:
def detect_postprocessing_full_scan(db_path, table, anchor_start,
                                     highflow_threshold=402,
                                     min_duration_min=2):
    """
    Scan the ENTIRE table (not just build windows) to detect the
    post-processing phase where HighFlowRAW rises above baseline.

    Strategy
    --------
    1. Load all rows from the table ordered by Timestamp.
    2. Apply a 30-second rolling median to smooth spikes.
    3. Find contiguous windows where HighFlowRAW > threshold.
    4. Filter out short bursts < min_duration_min.
    5. Return all valid segments (not just the largest) so you can
       inspect which one corresponds to post-processing.
    """
    sql = f"""
        SELECT Timestamp, ArgonFlowRAW, LowFlowRAW, HighFlowRAW, Power_W
        FROM   {table}
        ORDER  BY Timestamp
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(sql, conn, parse_dates=["Timestamp"])

    print(f"  Full table loaded — {len(df):,} rows  |  "
          f"{df['Timestamp'].min()}  →  {df['Timestamp'].max()}")

    anchor = pd.Timestamp(anchor_start)
    df["build_hours"] = (df["Timestamp"] - anchor).dt.total_seconds() / 3600

    # 30-second rolling median to remove single-sample spikes
    df = df.set_index("Timestamp")
    df["HighFlowRAW_smooth"] = (
        df["HighFlowRAW"]
          .rolling("30s", center=True)
          .median()
    )
    df = df.reset_index()

    # Flag rows above threshold
    df["active"]  = df["HighFlowRAW_smooth"] > highflow_threshold

    # Label contiguous active/inactive segments
    df["segment"] = (df["active"] != df["active"].shift()).cumsum()

    # Summarise each contiguous active segment
    active_rows = df[df["active"]].copy()

    if active_rows.empty:
        print(f"  No rows above threshold {highflow_threshold}. "
              "Try lowering highflow_threshold.")
        return None

    seg_summary = (
        active_rows.groupby("segment")
        .agg(
            seg_start    = ("Timestamp",      "min"),
            seg_end      = ("Timestamp",      "max"),
            peak_high    = ("HighFlowRAW",    "max"),
            mean_high    = ("HighFlowRAW",    "mean"),
            row_count    = ("Timestamp",      "count"),
            bh_start     = ("build_hours",    "min"),
            bh_end       = ("build_hours",    "max"),
        )
        .assign(
            duration_min = lambda x:
                (x["seg_end"] - x["seg_start"]).dt.total_seconds() / 60
        )
        .query("duration_min >= @min_duration_min")
        .sort_values("seg_start")
        .reset_index(drop=True)
    )

    print(f"\n  Found {len(seg_summary)} segment(s) above threshold "
          f"{highflow_threshold} lasting ≥ {min_duration_min} min:\n")
    print(seg_summary[[
        "seg_start", "seg_end", "duration_min",
        "peak_high", "mean_high", "row_count",
        "bh_start", "bh_end"
    ]].to_string(index=True))

    return df, seg_summary


# ── Scan full table ───────────────────────────────────────────────────────────
TABLE_FULL = "DoERun"      # <-- full table name

print("Scanning full table for HighFlowRAW activity ...\n")
full_df, seg_summary = detect_postprocessing_full_scan(
    db_path            = DB_PATH,
    table              = TABLE_FULL,
    anchor_start       = R1_ANCHOR,   # reference anchor for build_hours display
    highflow_threshold = 402,
    min_duration_min   = 2,
)

In [ ]:
# ── Preview the segment table ─────────────────────────────────────────────────
if seg_summary is not None:
    print("Segments detected (use the index to select):\n")
    display(seg_summary[[
        "seg_start", "seg_end", "duration_min", "peak_high", "bh_start", "bh_end"
    ]])

In [ ]:
# ── Set these based on the printed segment table ──────────────────────────────
R1_POSTPROC_IDX = 0    # <-- index of the post-processing segment for Run 1
R2_POSTPROC_IDX = 3    # <-- index of the post-processing segment for Run 2

def extract_segment(full_df, seg_summary, idx, anchor_start):
    """
    Slice the full dataframe for one detected segment and
    re-anchor build_hours to the given build start.
    """
    row       = seg_summary.iloc[idx]
    mask      = (
        (full_df["Timestamp"] >= row["seg_start"]) &
        (full_df["Timestamp"] <= row["seg_end"])
    )
    seg       = full_df[mask].copy()
    seg       = seg.drop(
        columns=[c for c in ["HighFlowRAW_smooth","active","segment"]
                 if c in seg.columns]
    )
    anchor    = pd.Timestamp(anchor_start)
    seg["build_hours"] = (
        seg["Timestamp"] - anchor
    ).dt.total_seconds() / 3600
    return seg, {
        "start"        : row["seg_start"],
        "end"          : row["seg_end"],
        "duration_min" : row["duration_min"],
    }


postproc_r1, meta_r1 = extract_segment(
    full_df, seg_summary, R1_POSTPROC_IDX, R1_ANCHOR
)
postproc_r2, meta_r2 = extract_segment(
    full_df, seg_summary, R2_POSTPROC_IDX, R2_ANCHOR
)

for label, seg, meta in [
    ("Post-proc R1", postproc_r1, meta_r1),
    ("Post-proc R2", postproc_r2, meta_r2),
]:
    print(f"{label}  |  {meta['start']}  →  {meta['end']}  "
          f"({meta['duration_min']:.1f} min)  |  "
          f"build hrs {seg['build_hours'].min():.3f} → "
          f"{seg['build_hours'].max():.3f}  |  "
          f"{len(seg):,} rows")

In [ ]:
def build_postproc_figure(df, meta, run_label, resample_s=30):
    """
    All four attributes in one single unified figure.
    Uses explicit yaxis / yaxis2 assignment instead of make_subplots
    to guarantee everything renders in one shared plot area.

    Left  Y-axis : ArgonFlowRAW, LowFlowRAW, HighFlowRAW
    Right Y-axis : Power_W
    """
    dur_min = meta["duration_min"]

    # ── Downsample if window is long enough ───────────────────────────────────
    if dur_min > 5:
        df_plot = (
            df.set_index("Timestamp")
              .resample(f"{resample_s}s")
              .mean()
              .reset_index()
        )
        bh_offset        = df["build_hours"].iloc[0]
        anchor           = df["Timestamp"].iloc[0]
        df_plot["build_hours"] = bh_offset + (
            df_plot["Timestamp"] - anchor
        ).dt.total_seconds() / 3600
        ds_note = f"{resample_s}s avg · {len(df_plot)} pts"
    else:
        df_plot = df.copy()
        ds_note = f"full resolution · {len(df_plot)} pts"

    x     = df_plot["build_hours"]
    x_min = x.min()
    x_max = x.max()

    # ── Adaptive x-axis ticks ─────────────────────────────────────────────────
    duration = x_max - x_min
    if duration <= 0.5:
        tick_step = 0.05
    elif duration <= 2.0:
        tick_step = 0.1
    elif duration <= 6.0:
        tick_step = 0.25
    else:
        tick_step = 1.0

    tick_vals = [
        round(t, 4)
        for t in np.arange(
            np.floor(x_min / tick_step) * tick_step,
            x_max + tick_step,
            tick_step,
        )
    ]
    tick_labels = [
        f"{t:.2f}" if tick_step < 1 else f"{int(round(t))}"
        for t in tick_vals
    ]

    # ── Build figure using explicit yaxis references ──────────────────────────
    fig = go.Figure()

    # HighFlowRAW — left axis, filled
    fig.add_trace(go.Scattergl(
        x         = x,
        y         = df_plot["HighFlowRAW"],
        name      = "HighFlowRAW",
        mode      = "lines",
        yaxis     = "y",
        line      = dict(color=PALETTE["HighFlowRAW"], width=2.2),
        fill      = "tozeroy",
        fillcolor = "rgba(6,214,160,0.07)",
        hovertemplate = (
            "<b>HighFlowRAW</b><br>"
            "Hour: %{x:.3f}<br>"
            "RAW: %{y:.1f}<extra></extra>"
        ),
    ))

    # ArgonFlowRAW — left axis
    fig.add_trace(go.Scattergl(
        x         = x,
        y         = df_plot["ArgonFlowRAW"],
        name      = "ArgonFlowRAW",
        mode      = "lines",
        yaxis     = "y",
        line      = dict(color=PALETTE["ArgonFlowRAW"], width=1.6),
        hovertemplate = (
            "<b>ArgonFlowRAW</b><br>"
            "Hour: %{x:.3f}<br>"
            "RAW: %{y:.1f}<extra></extra>"
        ),
    ))

    # LowFlowRAW — left axis
    fig.add_trace(go.Scattergl(
        x         = x,
        y         = df_plot["LowFlowRAW"],
        name      = "LowFlowRAW",
        mode      = "lines",
        yaxis     = "y",
        line      = dict(color=PALETTE["LowFlowRAW"], width=1.6),
        hovertemplate = (
            "<b>LowFlowRAW</b><br>"
            "Hour: %{x:.3f}<br>"
            "RAW: %{y:.1f}<extra></extra>"
        ),
    ))

    # Power_W — right axis
    fig.add_trace(go.Scattergl(
        x         = x,
        y         = df_plot["Power_W"],
        name      = "Power_W",
        mode      = "lines",
        yaxis     = "y2",
        line      = dict(color=PALETTE["Power_W"], width=1.6),
        hovertemplate = (
            "<b>Power_W</b><br>"
            "Hour: %{x:.3f}<br>"
            "Power: %{y:.0f} W<extra></extra>"
        ),
    ))

    # ── Compute Y ranges ──────────────────────────────────────────────────────
    gas_cols = ["ArgonFlowRAW", "LowFlowRAW", "HighFlowRAW"]
    gas_min  = df_plot[gas_cols].min().min()
    gas_max  = df_plot[gas_cols].max().max()
    gas_pad  = (gas_max - gas_min) * 0.10
    pwr_max  = df_plot["Power_W"].max()

    # ── Hour reference lines ──────────────────────────────────────────────────
    for t in tick_vals:
        fig.add_vline(
            x          = t,
            line_dash  = "dot",
            line_color = "#2a3040",
            line_width = 1,
            opacity    = 0.5,
        )

    # ── Full layout with explicit yaxis and yaxis2 ────────────────────────────
    fig.update_layout(
        title=dict(
            text=(
                f"<b>{run_label} — Post-Processing Phase</b><br>"
                f"<sup>{meta['start']}  →  {meta['end']}  ·  "
                f"{dur_min:.1f} min  ·  {ds_note}</sup>"
            ),
            font    = dict(color=TEXT, size=14),
            x       = 0.01,
            xanchor = "left",
        ),
        plot_bgcolor  = BG,
        paper_bgcolor = BG,
        font          = dict(color=TEXT, family="Segoe UI, Arial, sans-serif", size=12),
        hovermode     = "x unified",
        hoverlabel    = dict(
            bgcolor     = PANEL_BG,
            bordercolor = "#3a4560",
            font        = dict(size=11, color=TEXT),
        ),
        legend=dict(
            bgcolor         = PANEL_BG,
            bordercolor     = "#2a3040",
            borderwidth     = 1,
            font            = dict(size=11),
            orientation     = "h",
            x               = 0,
            y               = -0.20,
            itemclick       = "toggle",
            itemdoubleclick = "toggleothers",
        ),
        margin = dict(l=72, r=80, t=90, b=110),
        height = 520,

        # ── X-axis ────────────────────────────────────────────────────────────
        xaxis=dict(
            title_text = "Build Hours",
            range      = [x_min - tick_step * 0.3, x_max + tick_step * 0.3],
            tickmode   = "array",
            tickvals   = tick_vals,
            ticktext   = tick_labels,
            tickangle  = -35,
            showgrid   = True,
            gridcolor  = GRID,
            gridwidth  = 1,
            zerolinecolor  = "#2a3040",
            tickcolor  = "#4a5568",
            linecolor  = "#2a3040",
            tickfont   = dict(color=TEXT, size=11),
            title_font = dict(color=AXIS_LBL, size=12),
        ),

        # ── Left Y-axis : Gas RAW ─────────────────────────────────────────────
        yaxis=dict(
            title_text = "Gas Sensor RAW (ADC counts)",
            range      = [gas_min - gas_pad, gas_max + gas_pad],
            showgrid   = True,
            gridcolor  = GRID,
            gridwidth  = 1,
            zerolinecolor  = "#2a3040",
            tickcolor  = "#4a5568",
            linecolor  = "#2a3040",
            tickfont   = dict(color=TEXT, size=11),
            title_font = dict(color=AXIS_LBL, size=12),
        ),

        # ── Right Y-axis : Power ──────────────────────────────────────────────
        yaxis2=dict(
            title_text = "Power_W  (W)",
            range      = [0, pwr_max * 1.12],
            overlaying = "y",
            side       = "right",
            showgrid   = False,
            tickcolor  = "#4a5568",
            linecolor  = "#2a3040",
            tickfont   = dict(color=TEXT, size=11),
            title_font = dict(color=AXIS_LBL, size=12),
        ),
    )

    return fig


print("✓ Post-processing figure builder ready.")

In [ ]:
fig_postproc_r1 = build_postproc_figure(
    df         = postproc_r1,
    meta       = meta_r1,
    run_label  = "Run 1",
    #resample_s = 30,
)
fig_postproc_r1.show()

print("Exporting run1 postprocessing...")
export_fig(fig_postproc_r2, "run1_postprocessing", h=560)

In [ ]:
fig_postproc_r2 = build_postproc_figure(
    df         = postproc_r2,
    meta       = meta_r2,
    run_label  = "Run 2",
    #resample_s = 30,
)
fig_postproc_r2.show()

print("Exporting run2 postprocessing...")
export_fig(fig_postproc_r2, "run2_postprocessing", h=560)